# Experiment 1 Runner

This notebook runs `experiment_1/train_experiment_1.py` using the same clone-or-pull workflow used elsewhere in this repo.

In [ ]:
# Set RUN_ENV to 'local' on your computer or 'colab' in Google Colab.
RUN_ENV = 'colab'

GIT_REPO_URL = 'https://github.com/ffbskt/AgentAI.git'
LOCAL_REPO_DIR = r'D:/Codex projects/Transformer_Graph3/arithmetic-transformer'
COLAB_PROJECT_ROOT = '/content/drive/MyDrive/AgentAI'

In [ ]:
if RUN_ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if RUN_ENV == 'colab':
    project_root = Path(COLAB_PROJECT_ROOT)
    repo_root = project_root
    git_dir = repo_root / '.git'

    if git_dir.exists():
        print(f"'{repo_root}' is already a git repository. Pulling latest changes...")
        subprocess.run(['git', '-C', str(repo_root), 'pull'], check=True)
    elif repo_root.exists():
        print(f"'{repo_root}' exists but is not a git repository. Removing and re-cloning...")
        shutil.rmtree(repo_root)
        project_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GIT_REPO_URL, str(repo_root)], check=True)
    else:
        print(f"Cloning '{GIT_REPO_URL}' into '{repo_root}'...")
        project_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GIT_REPO_URL, str(repo_root)], check=True)

    REPO_DIR = repo_root / 'arithmetic-transformer'
else:
    REPO_DIR = Path(LOCAL_REPO_DIR)

EXPERIMENT_DIR = REPO_DIR / 'experiment_1'
LOG_DIR = EXPERIMENT_DIR / 'logs' / 'colab_run'
LOG_DIR.mkdir(parents=True, exist_ok=True)

print('Repo dir:', REPO_DIR)
print('Experiment dir:', EXPERIMENT_DIR)
print('Log dir:', LOG_DIR)

In [ ]:
%cd {REPO_DIR}
!python -m pip install -q -r requirements-colab.txt

In [ ]:
import torch
import json

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = 'logs/colab_run'

# Edit these settings directly in the notebook when running in Colab.
EXPERIMENT_CONFIG = {
    'model': {
        'kind': 'transformer-sine',
        'hidden_size': 64,
        'ffw_size': 128,
        'num_layers': 2,
        'num_heads': 4,
        'dropout': 0.0,
        'lr': 0.001
    },
    'train_defaults': {
        'epochs': 1,
        'batch_size': 64,
        'train_samples': 256,
        'val_samples': 128,
        'test_samples': 128,
        'max_new_tokens': 48,
        'rl_steps': 4,
        'rl_batch_size': 16,
        'num_generations': 4,
        'temperature': 1.0,
        'kl_coef': 0.02,
        'best_of_n_steps': 2,
        'seed': 11
    },
    'phases': [
        {'name': 'step1_1d', 'description': '1-digit warmup', 'fmt': 'A', 'shape': '1d+1d', 'carry_mode': 'any', 'rl_enabled': True},
        {'name': 'step2_2d', 'description': '2-digit reasoning', 'fmt': 'C', 'shape': '2d+2d', 'carry_mode': 'simple', 'rl_enabled': True},
        {'name': 'step3_3d', 'description': '3-digit reasoning', 'fmt': 'C', 'shape': '3d+3d', 'carry_mode': 'any', 'rl_enabled': True},
        {'name': 'step4_4d', 'description': '4-digit reasoning', 'fmt': 'C', 'shape': '4d+4d', 'carry_mode': 'any', 'rl_enabled': True},
        {'name': 'step5_5d', 'description': '5-digit reasoning', 'fmt': 'C', 'shape': '5d+5d', 'carry_mode': 'heavy', 'rl_enabled': True}
    ]
}

CONFIG_JSON = json.dumps(EXPERIMENT_CONFIG)

print('Selected device:', DEVICE)
print('Phases:', [phase['name'] for phase in EXPERIMENT_CONFIG['phases']])
cmd = f'''python train_experiment_1.py \
  --device {DEVICE} \
  --config-json '{CONFIG_JSON}' \
  --output-dir {OUTPUT_DIR}'''

print(cmd)

In [ ]:
%cd {EXPERIMENT_DIR}
!{cmd}

In [ ]:
!ls -lah "$LOG_DIR"
!cat "$LOG_DIR/summary.csv" || true